# Metis — Train from a text dataset

Train a Μῆτις language model the **normal way**: next-token prediction on a
plain-text corpus you provide. No teacher API, no tunnel — just your data
and the GPU.

The trainer runs a fixed number of steps (`--iters`), saves checkpoints to
Drive, and `--resume` continues a stopped run.

---
## Step 1 — Mount Google Drive
Your dataset and checkpoints live here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 — Clone repo + install dependencies

In [ ]:
import os, subprocess

REPO = "https://github.com/iamasrakib/Metis.git"
METIS_DIR = "/content/Metis"

if os.path.isdir(METIS_DIR):
    subprocess.run(['git', '-C', METIS_DIR, 'pull', '--quiet'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', REPO, METIS_DIR], check=True)
os.chdir(METIS_DIR)

# datasets / huggingface_hub: FineWeb-Edu download. bitsandbytes: 8-bit Adam
# (fits the ~1B model in the T4's 16 GB). tiktoken: cl100k_base tokenizer.
subprocess.run(['pip', 'install', '-q', 'numpy', 'tqdm', 'tiktoken', 'tokenizers',
                'datasets', 'huggingface_hub', 'bitsandbytes'], check=True)

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
print(f'Done. GPU: {gpu}')

---
## Step 3 — Download FineWeb-Edu (~500M tokens)
Streams the first ~2 GB of FineWeb-Edu (`sample-10BT` — HuggingFace's
high-quality educational web corpus) into `MyDrive/Metis/fineweb_edu.txt`.
≈ 500M tokens at ~4 chars/token. Runs once; re-running skips if the file
is already there.

In [ ]:
import os, subprocess

TARGET_CHARS = 2_000_000_000   # ≈ 500M tokens at ~4 chars/token
DEST = "/content/drive/MyDrive/Metis/fineweb_edu.txt"

if os.path.exists(DEST) and os.path.getsize(DEST) >= TARGET_CHARS:
    print(f"Dataset already downloaded: {DEST} ({os.path.getsize(DEST):,} chars)")
else:
    try:
        from datasets import load_dataset
    except ImportError:
        subprocess.run(['pip', 'install', '-q', 'datasets', 'huggingface_hub'], check=True)
        from datasets import load_dataset

    os.makedirs(os.path.dirname(DEST), exist_ok=True)
    print(f"Streaming FineWeb-Edu (sample-10BT) → {DEST} ...")
    ds = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT",
                      split="train", streaming=True)
    n_chars, n_docs = 0, 0
    with open(DEST, "w", encoding="utf-8") as f:
        for doc in ds:
            text = doc["text"].strip()
            if not text:
                continue
            f.write(text + "\n\n")
            n_chars += len(text)
            n_docs += 1
            if n_docs % 5000 == 0:
                print(f"  {n_chars/1e6:,.0f}M chars after {n_docs:,} docs")
            if n_chars >= TARGET_CHARS:
                break
    print(f"Done: {n_chars/1e6:,.0f}M chars in {n_docs:,} docs → {DEST}")
    print(f"≈ {n_chars/4e6:,.0f}M tokens (char estimate, ~4 chars/token)")

---
## Step 4 — Pick your dataset
Defaults to the FineWeb-Edu corpus downloaded in Step 3. You can instead
point at your own `.txt` corpus in Drive (or upload it here), or set a
Colab secret named `DATASET`.

In [ ]:
import os
from google.colab import userdata

# ── Your dataset ──────────────────────────────────────────────────────────
# Default: the FineWeb-Edu corpus downloaded in Step 3.
# Option A: put a plain-text corpus (.txt) in Drive and set the path below.
# Option B: upload it to this runtime with the 📂 Files sidebar and set the
#           path to the /content/… location.
# Option C: set a Colab secret named DATASET (key icon → + New secret).
DRIVE_DATASET = "/content/drive/MyDrive/Metis/fineweb_edu.txt"

# Fallback so the notebook runs out of the box if no dataset is found.
# data/sample.txt is a small corpus shipped with the repo.
REPO_FALLBACK = "data/sample.txt"

def _secret(name, default=""):
    try:
        return userdata.get(name)
    except Exception:
        return default

DATASET = _secret("DATASET", "") or DRIVE_DATASET
if not os.path.isfile(DATASET):
    print(f"{DATASET} not found — falling back to repo corpus {REPO_FALLBACK}")
    DATASET = REPO_FALLBACK

print(f"Dataset: {DATASET} ({os.path.getsize(DATASET):,} bytes)")

---
## Step 5 — Start training (~1B model on FineWeb-Edu)
Runs `metis train` with the `1b` preset. Ctrl+C stops it (saves first);
re-running the cell resumes from the last checkpoint.
- `--optimizer bnb8bit` (bitsandbytes 8-bit Adam) is what fits the ~1B
  model in the T4's 16 GB VRAM.
- `--no-cuda-graphs` is required: CUDA-graph capture is non-checkpointed
  and would OOM a 1B model — the eager loop keeps gradient checkpointing.
- Expect ~1.5–3k tokens/s on a T4 → the 4000 steps (≈525M tokens) take
  several Colab sessions. Just re-run this cell to continue.

In [ ]:
import os, subprocess

# Link checkpoints AND the tokenization cache to Drive, so both survive Colab
# disconnects — resume never re-tokenizes the 2 GB corpus.
DRIVE_BASE = "/content/drive/MyDrive/Metis"
for folder in ("checkpoints_1b", "cache"):
    drive_path = os.path.join(DRIVE_BASE, folder)
    os.makedirs(drive_path, exist_ok=True)
    if not os.path.lexists(folder):
        os.symlink(drive_path, folder)
        print(f"Linked {folder} -> {drive_path}")

# Train the ~1B model on FineWeb-Edu. Ctrl+C to stop (saves first); re-run
# this cell to resume from where it stopped.
result = subprocess.run(
    ["python", "-u", "-m", "metis.cli", "train",
     "--checkpoint-dir", "checkpoints_1b",
     "--dataset", DATASET,
     "--preset", "1b",
     "--optimizer", "bnb8bit",
     "--no-cuda-graphs",
     "--tokenizer", "cl100k_base",
     "--iters", "4000",
     "--batch-size", "8",
     "--grad-accum", "16",
     "--seq-len", "1024",
     "--resume"]
)
if result.returncode != 0:
    print(f"\nTraining exited with error code {result.returncode}")
    print("Check the traceback above — most common causes:")
    print("  • wrong --dataset path")
    print("  • OOM on the free T4 → lower --batch-size to 4 (or --grad-accum to 8)")

---
**Stop:** `Ctrl+C` in the run cell (the checkpoint saves first).
**Resume:** re-run the cell — `--resume` picks up from the last checkpoint
in `MyDrive/Metis/checkpoints_1b`.
**Train longer:** raise `--iters` in Step 5. Each re-run continues exactly
where it stopped.
**OOM?** Lower `--batch-size` to `4` (and/or `--grad-accum` to `8`).
**Test the model:** copy the `checkpoints_1b` folder to your PC and run
`metis chat --checkpoint-dir checkpoints_1b` (or `metis generate`).
**Old models:** delete `MyDrive/Metis/checkpoints_train/` (and any old
dataset files) when you're done — this run uses the fresh
`checkpoints_1b` folder.